# Лабораторная работа 1:  уровень решения проблемы обработки и генерации естественного языка ч.1

1. **Токенизация** произвольного документа;
2. **Нормализация** словаря — стемминг (Snowball), лемматизация (pymorphy3),
   синонимия, удаление стоп-слов;
3. **Анализатор тональности** — мультиномиальный наивный байесовский
   классификатор (3 класса: negative / neutral / positive), реализованный с нуля
   на стандартной библиотеке.

## 1. Постановка задачи

Для набора русскоязычных отзывов требуется:

- разбить текст на токены и предложения;
- нормализовать словарь: леммы/стемы, синонимы к канонической форме, стоп-слова;
- построить классификатор тональности по трём классам (negative / neutral / positive);
- оценить качество на отложенной выборке.


## 2. Окружение и импорт

In [ ]:
import os, sys, glob

_here = os.getcwd()
_root = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
sys.path.insert(0, _root)

from src import tokenize, sent_tokenize, normalize_text, canonical_tokens, build_vocabulary
from src.sentiment import SentimentAnalyzer
from src.data import download_rureviews, load_corpus, corpus_stats, CLASS_LABELS

## 3. Данные: датасет RuReviews

**RuReviews** ([Smetanin & Komarov, 2019](https://ieeexplore.ieee.org/document/8807792)) —
русскоязычные отзывы о товарах с тремя сбалансированными классами тональности.

- Зеркало: <https://github.com/sismetanin/rureviews>, файл
  `women-clothing-accessories.3-class.balanced.csv` (TSV, колонки `review<TAB>sentiment`).
- В исходном файле нейтральный класс записан с опечаткой `neautral` — загрузчик
  приводит его к каноническому `neutral`.



In [ ]:
corpus_full = os.path.join(_root, "data", "rureviews.csv")
try:
    download_rureviews(corpus_full)
    corpus_path = corpus_full
except RuntimeError as exc:
    print("Сеть недоступна — используем встроенную компактную выборку.")
    print(exc)
    corpus_path = os.path.join(_root, "src", "resources", "train_data.csv")

rows = load_corpus(corpus_path)
counts = corpus_stats(rows)
print("Корпус:", os.path.relpath(corpus_path, _root))
print("Всего документов:", len(rows))
for label in CLASS_LABELS:
    print(f"  {label}: {counts[label]}")

In [ ]:
seen = {label: None for label in CLASS_LABELS}
for label, text in rows:
    if seen[label] is None:
        seen[label] = text
    if all(v is not None for v in seen.values()):
        break

for label in CLASS_LABELS:
    sample = seen[label] or ""
    preview = sample[:180] + ("…" if len(sample) > 180 else "")
    print(f"[{label}]")
    print("  " + preview)
    print()

## 4. Токенизация

`tokenize` извлекает слова (кириллица а–я + ё, латиница a–z), допускает внутренние
дефис и апостроф; цифры и пунктуация отбрасываются, поэтому цифра «2» играет роль
разделителя. `sent_tokenize` делит текст на предложения по `.`, `!`, `?`, `…` и
переводам строк.

In [4]:
text = "Купил смартфон — экран отличный, батарея держит 2 дня! Рекомендую."
print("Предложения:", sent_tokenize(text))
print()
print("Токены:", tokenize(text))

Предложения: ['Купил смартфон — экран отличный, батарея держит 2 дня!', 'Рекомендую.']

Токены: ['купил', 'смартфон', 'экран', 'отличный', 'батарея', 'держит', 'дня', 'рекомендую']


## 5. Нормализация: стемминг, лемматизация, синонимия, стоп-слова

Каждый токен проходит: нижний регистр → стемминг (`SnowballStemmer('russian')`) →
лемматизация (`pymorphy3.MorphAnalyzer`) → приведение синонимов к канонической
лемме (словарь `resources/synonyms_ru.json`). Стоп-слова (`resources/stopwords_ru.txt`)
помечаются флагом `is_stopword` и исключаются из признаков классификатора.



In [ ]:
sample = "Это были отличные, замечательные фильмы, но цена недешёвая."
for t in normalize_text(sample):
    print(f"{t.original:14s} stem={t.stem:12s} lemma={t.lemma:12s} canon={t.canonical:12s} stop={t.is_stopword}")
print()
print("Признаки (канон. формы без стоп-слов):", canonical_tokens(sample))

## 6. Нормализованный словарь

`build_vocabulary` строит частотный словарь канонических лемм (без стоп-слов) по
набору документов, отсортированный по убыванию частоты.

In [6]:
docs = [open(f, encoding="utf-8").read() for f in sorted(glob.glob(os.path.join(_root, "data", "sample_docs", "*.txt")))]
vocab = build_vocabulary(docs)
print("Размер словаря:", len(vocab))
for w, c in list(vocab.items())[:20]:
    print(f"  {w}: {c}")

Размер словаря: 83
  день: 4
  делать: 3
  камера: 3
  корпус: 3
  смартфон: 3
  снимок: 3
  устройство: 3
  батарея: 2
  деньга: 2
  интерфейс: 2
  использование: 2
  купить: 2
  назад: 2
  неделя: 2
  обычный: 2
  работать: 2
  рекомендовать: 2
  цвет: 2
  экран: 2
  активный: 1


## 7. Обучение модели (наивный Байес)

Мультиномиальный наивный Байес оценивает апостериорную вероятность класса:

    P(c | d) ∝ P(c) · ∏ₜ P(t | c)

где `t` пробегает канонические токены документа `d`. Приор `P(c)` — доля класса в
обучающей выборке, а условные вероятности сглаживаются по Лапласу (add-α):

    P̂(t | c) = (count(t, c) + α) / (count(c) + α · |V|)

`|V|` — размер словаря, `α = 1.0`. Для численной устойчивости вычисляются
логарифмы вероятностей (log-пространство); токен вне словаря (OOV) даёт нулевой
вклад. Решение — класс с максимальной апостериорной вероятностью.

`train(...)` выполняет весь конвейер: загрузка корпуса → стратифицированное
разбиение 80/20 (seed 42) → построение признаков → отбор токенов по документной
частоте `min_df` → fit → сохранение модели → accuracy. Порог `min_df` адаптивен
(`max(1, min(min_df, n_train//10))`), поэтому малые корпуса не обнуляются.

In [ ]:
analyzer = SentimentAnalyzer()
stats = analyzer.train(
    corpus_csv=corpus_path,
    out_model=os.path.join(_root, "src", "models", "sentiment_model.json"),
    limit=15000,
    alpha=1.0,
    test_size=0.2,
    min_df=3,
)
print("accuracy   :", round(stats["accuracy"], 4))
print("n_train    :", stats["n_train"])
print("n_test     :", stats["n_test"])
print("vocab_size :", analyzer.model.vocab_size)

## 8. Оценка и предсказания

Метрика — accuracy на стратифицированной отложенной выборке (20%). Ниже — примеры
предсказаний обученной модели на коротких фразах и на файлах-документах.

In [8]:
examples = [
    "Отличный товар, очень доволен, рекомендую!",
    "Ужасный товар, сплошной брак, разочарован.",
    "Обычный товар, стандартное качество.",
]
for e in examples:
    r = analyzer.analyze(e)
    probs = {k: round(v, 3) for k, v in r.probabilities.items()}
    print(f"«{e}»")
    print(f"   -> {r.label}  {probs}")
    print()

for f in sorted(glob.glob(os.path.join(_root, "data", "sample_docs", "*.txt"))):
    r = analyzer.analyze_file(f)
    probs = {k: round(v, 3) for k, v in r.probabilities.items()}
    print(f"{os.path.basename(f):15s} -> {r.label:8s} {probs}")

«Отличный товар, очень доволен, рекомендую!»
   -> positive  {'negative': 0.036, 'neutral': 0.009, 'positive': 0.955}

«Ужасный товар, сплошной брак, разочарован.»
   -> negative  {'negative': 0.833, 'neutral': 0.166, 'positive': 0.001}

«Обычный товар, стандартное качество.»
   -> neutral  {'negative': 0.334, 'neutral': 0.504, 'positive': 0.162}

mixed.txt       -> neutral  {'negative': 0.0, 'neutral': 0.999, 'positive': 0.001}
negative.txt    -> negative {'negative': 0.979, 'neutral': 0.021, 'positive': 0.0}
positive.txt    -> positive {'negative': 0.0, 'neutral': 0.0, 'positive': 1.0}


## 9. Пользовательские данные

Модель переобучается на произвольном корпусе CSV `label,text` (label ∈ {negative,
neutral, positive}, все три класса обязательны). Ниже — мини-корпус из 24 отзывов
(8 на класс): обучение и два предсказания. На реальных данных используйте тысячи
сбалансированных строк.

In [ ]:
custom_csv = os.path.join(_root, "data", "custom_reviews_demo.csv")
lines = ["label,text"]
pos = ["Отличный товар, очень доволен, рекомендую",
       "Качество супер, всё понравилось, советую",
       "Прекрасная вещь, сидит отлично, рекомендую",
       "Очень хороший товар, качество отличное",
       "Доволен покупкой, всё замечательно",
       "Рекомендую, товар отличный и недорогой",
       "Всё понравилось, качество хорошее",
       "Отличная покупка, всем советую"]
neg = ["Ужасный товар, сплошной брак, не советую",
       "Качество плохое, разочарован, зря купил",
       "Отвратительное качество, не рекомендую",
       "Всё плохо, товар ужасный, вернул",
       "Не советую, брак и плохое качество",
       "Разочарован, деньги на ветер, качество ужасное",
       "Товар плохой, не рекомендую никому",
       "Ужасно, качество отвратительное"]
neu = ["Обычный товар, ничего особенного",
       "Среднее качество, могло быть лучше",
       "Нормально, своё дело делает",
       "Ничего выдающегося, обычная вещь",
       "Товар как товар, без восторгов",
       "Средненько, за свою цену нормально",
       "Обычно, ожидал большего",
       "Нейтрально, без претензий"]
for t in pos:
    lines.append("positive," + t)
for t in neg:
    lines.append("negative," + t)
for t in neu:
    lines.append("neutral," + t)
with open(custom_csv, "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

a2 = SentimentAnalyzer()
metrics = a2.train(custom_csv, os.path.join(_root, "src", "models", "custom_model.json"))
print("Метрики обучения:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()})
for e in ["Сервис отличный, рекомендую!", "Ужасное качество, не советую"]:
    r = a2.analyze(e)
    probs = {k: round(v, 3) for k, v in r.probabilities.items()}
    print(f"  «{e}» -> {r.label}  {probs}")

## 10. Выводы

- Реализован полный конвейер: токенизация → нормализация → наивный Байес.
- На сбалансированной выборке RuReviews (15 000 отзывов, 12 000 / 3 000) accuracy ≈ 0.68.
- Модель детерминирована (seed 42) и сериализуется в переносимый JSON.
- Точность можно повысить: больше данных, n-граммы признаков, TF-IDF, учёт отрицаний.